# Prompt Engineering Notebook for Steer/Exec Intervention Augmentation

This notebook is for iterating on the prompt templates before running a larger augmentation batch.

It uses the same files as the batch script:

- `interventions.json`
- `prompts/*.md`
- `src/trace_augmentor.py`

The basic loop is:

1. load real interleaved traces,
2. choose an intervention and cut point,
3. render the exact prompt,
4. call OpenRouter with structured outputs if a key is set,
5. validate the returned blocks,
6. inspect whether the intervention feels natural.

The notebook is deliberately lightweight so you can edit prompt files on disk and re-run cells.

Set `STEER_EXEC_INPUT_PATH` to override the default trace source.


In [ ]:
import os
from pathlib import Path


def _load_env_file(path: str | Path) -> dict[str, str]:
    values: dict[str, str] = {}
    p = Path(path)
    if not p.exists():
        return values
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip().strip('"').strip("'")
    return values


env_path = Path("BuildSFTDataset/.env")
env = _load_env_file(env_path)

# Prefer canonical OpenRouter variable, fallback to existing naming in this repo
api_key = (
    env.get("OPENROUTER_API_KEY")
    or os.environ.get("OPENROUTER_API_KEY")
    or env.get("OPEN_ROUTER_KEY")
    or os.environ.get("OPEN_ROUTER_KEY")
)

if not api_key:
    raise ValueError(f"No OpenRouter key found in env vars or {env_path}")

os.environ["OPENROUTER_API_KEY"] = api_key
print("OPENROUTER_API_KEY loaded:", bool(api_key))


In [ ]:
import asyncio
import importlib
import inspect
from pathlib import Path
import json
import os
import random
import sys
from typing import Any, Dict

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import src.trace_augmentor as trace_augmentor_module

trace_augmentor_module = importlib.reload(trace_augmentor_module)

TokenCounter = trace_augmentor_module.TokenCounter
OpenRouterAsyncClient = trace_augmentor_module.OpenRouterAsyncClient
OpenRouterClient = trace_augmentor_module.OpenRouterClient
augment_record = trace_augmentor_module.augment_record
choose_intervention = trace_augmentor_module.choose_intervention
parse_trace_text = trace_augmentor_module.parse_trace_text
pair_count = trace_augmentor_module.pair_count
slice_pairs = trace_augmentor_module.slice_pairs
select_cut_pair_index = trace_augmentor_module.select_cut_pair_index
build_prompt_values = trace_augmentor_module.build_prompt_values
render_prompt = trace_augmentor_module.render_prompt
choose_prompt_path = trace_augmentor_module.choose_prompt_path
build_intervention_schema = trace_augmentor_module.build_intervention_schema
validate_generated_window = trace_augmentor_module.validate_generated_window
render_trace = trace_augmentor_module.render_trace
render_window = trace_augmentor_module.render_window
get_next_original_context = trace_augmentor_module.get_next_original_context
maybe_call_bridge_judge = trace_augmentor_module.maybe_call_bridge_judge
extract_trace_text = trace_augmentor_module.extract_trace_text
load_json = trace_augmentor_module.load_json
load_jsonl = trace_augmentor_module.load_jsonl


def coalesce_record_user_prompt(record: Dict[str, Any]) -> str:
    """Extract a displayable user prompt from either dataset shape."""
    if isinstance(record.get("user_prompt"), str):
        return record["user_prompt"]

    messages = record.get("messages")
    if isinstance(messages, list) and messages:
        first_message = messages[0]
        if isinstance(first_message, dict):
            message_content = first_message.get("content")
            if isinstance(message_content, str):
                return message_content
    return ""


def coalesce_record_task_id(record: Dict[str, Any]) -> str:
    """Return a display id for the record from known fields."""
    if isinstance(record.get("task_id"), str):
        return record["task_id"]
    if isinstance(record.get("id"), str):
        return record["id"]
    return ""


def trace_text_from_record(record: Dict[str, Any]) -> str:
    """Return the model trace text from a record in any supported format."""
    if any(k in record for k in ("trace", "trace_blocks", "assistant")):
        return extract_trace_text(record)

    messages = record.get("messages")
    if isinstance(messages, list) and messages:
        assistant_messages = [
            item
            for item in messages
            if isinstance(item, dict)
            and item.get("role") == "assistant"
            and isinstance(item.get("content"), str)
        ]
        if assistant_messages:
            return assistant_messages[-1]["content"]
    raise KeyError(
        "Record must contain one of: trace, trace_blocks, assistant, messages[role=assistant]"
    )


def make_openrouter_client() -> OpenRouterAsyncClient:
    """Build a notebook client using the currently reloaded bundle module."""
    client_kwargs = {
        "api_key": OPENROUTER_API_KEY,
        "model": OPENROUTER_MODEL,
        "use_response_healing": OPENROUTER_USE_RESPONSE_HEALING,
        "provider_data_collection": "deny",
        "temperature": 0.4,
        "max_tokens": OPENROUTER_MAX_TOKENS,
    }
    signature = inspect.signature(OpenRouterAsyncClient)
    if "reasoning_effort" in signature.parameters:
        client_kwargs["reasoning_effort"] = OPENROUTER_REASONING_EFFORT
    return OpenRouterAsyncClient(**client_kwargs)

## Configuration

Set the paths, intervention model, and optional tokenizer for exec-length checks.

If `OPENROUTER_API_KEY` is not set, the notebook still works for prompt rendering and dry inspection.

In [ ]:
REPO_ROOT = ROOT.parent.parent

INTERVENTIONS_PATH = ROOT / "interventions.json"
PROMPTS_DIR = ROOT / "prompts"
SAMPLES_PATH = ROOT / "examples" / "sample_traces.jsonl"
REAL_TRACES_PATH = REPO_ROOT / "output" / "transformed_output.jsonl"
INPUT_TRACES_PATH = Path(os.environ.get("STEER_EXEC_INPUT_PATH", str(REAL_TRACES_PATH)))
MAX_RECORDS = int(os.environ.get("STEER_EXEC_MAX_RECORDS", "10"))

OPENROUTER_MODEL = "openai/gpt-oss-20b"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_REASONING_EFFORT = os.environ.get("OPENROUTER_REASONING_EFFORT", "low")
OPENROUTER_MAX_TOKENS = int(os.environ.get("OPENROUTER_MAX_TOKENS", "2400"))
OPENROUTER_USE_RESPONSE_HEALING = (
    os.environ.get("OPENROUTER_USE_RESPONSE_HEALING", "1") == "1"
)

TOKENIZER_NAME = None  # e.g. "meta-llama/Llama-3.1-8B-Instruct"
EXEC_TOKEN_LIMIT = 512
STYLE_WINDOW_PAIRS = 2

rng = random.Random(7)
token_counter = TokenCounter(TOKENIZER_NAME)

assert INPUT_TRACES_PATH.exists(), f"Input trace file not found: {INPUT_TRACES_PATH}"
interventions_obj = load_json(INTERVENTIONS_PATH)
records = load_jsonl(INPUT_TRACES_PATH)
if MAX_RECORDS > 0:
    records = records[:MAX_RECORDS]
len(records)

## Inspect the sample records

In [ ]:
for i, record in enumerate(records):
    print("=" * 80)
    print(i, coalesce_record_task_id(record))
    print(coalesce_record_user_prompt(record))
    print()
    trace_text = trace_text_from_record(record)
    print(trace_text[:600], "..." if len(trace_text) > 600 else "")
    print()

## Pick an example and an intervention

You can either:
- fix the intervention name manually, or
- let the notebook sample one at random.

The rendered prompt will include the next original steer for insert and bridge modes.

In [ ]:
record = records[0]
fixed_intervention_name = None  # e.g. "Check for any contradictions"

intervention_spec = choose_intervention(
    interventions_obj, rng, name=fixed_intervention_name
)
all_blocks = parse_trace_text(trace_text_from_record(record))
cut_after_pairs = select_cut_pair_index(
    total_pairs=pair_count(all_blocks),
    preferred_slots=intervention_spec.get("preferred_slots", []),
    rng=rng,
)
prefix_blocks = slice_pairs(all_blocks, cut_after_pairs)

intervention_spec["name"], cut_after_pairs, pair_count(all_blocks)

## Choose mode, variant, and pair count manually for inspection

In [ ]:
mode = intervention_spec["recommended_editor_mode"]
variant = intervention_spec["variants"][0]
k_pairs = intervention_spec["pairs_to_generate"]["default"]

mode, variant, k_pairs

In [ ]:
values = build_prompt_values(
    record=record,
    all_blocks=all_blocks,
    prefix_blocks=prefix_blocks,
    intervention_spec=intervention_spec,
    intervention_variant=variant,
    pairs_to_generate_k=k_pairs,
    exec_token_limit=EXEC_TOKEN_LIMIT,
    style_window_pairs=STYLE_WINDOW_PAIRS,
    validation_feedback="",
)

prompt_path = choose_prompt_path(PROMPTS_DIR, mode)
system_prompt = (PROMPTS_DIR / "system.md").read_text(encoding="utf-8")
user_prompt = render_prompt(prompt_path, values)

print("SYSTEM PROMPT:")
print(system_prompt[:1800], "..." if len(system_prompt) > 1800 else "")
print("\n" + "=" * 80 + "\n")
print("USER PROMPT:")
print(user_prompt[:4000], "..." if len(user_prompt) > 4000 else "")

## Call OpenRouter with structured outputs

This cell is optional. It uses the same JSON schema as the batch script and validates the returned blocks locally.

If the API key is missing, the cell will skip the call.

In [ ]:
if not OPENROUTER_API_KEY:
    print("OPENROUTER_API_KEY is not set; skipping live model call.")
else:
    client = make_openrouter_client()
    raw = await client.chat_json(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        json_schema=build_intervention_schema(k_pairs),
    )
    print(json.dumps(raw, indent=2, ensure_ascii=False))

## Validate the returned intervention window

In [ ]:
try:
    raw
except NameError:
    raw = {
        "blocks": [
            {"type": "steer", "text": variant},
            {
                "type": "exec",
                "text": "Mock block for notebook validation when no API key is configured.",
            },
        ]
    }

generated_blocks, errors = validate_generated_window(
    raw,
    requested_pairs=k_pairs,
    required_first_steer=variant,
    token_counter=token_counter,
    exec_token_limit=EXEC_TOKEN_LIMIT,
)

print("Errors:", errors)
print()
print(render_window(generated_blocks))

## Inspect bridge compatibility

For bridge prompts, this checks whether the old next steer still fits after the inserted window.

In [ ]:
next_original_steer, next_original_exec_preview = get_next_original_context(
    all_blocks, cut_after_pairs
)

if mode != "bridge":
    print("Current mode is not bridge. Set mode = 'bridge' above to use this cell.")
else:
    if OPENROUTER_API_KEY:
        judge = await maybe_call_bridge_judge(
            openrouter=client,
            prompts_dir=PROMPTS_DIR,
            trace_prefix=render_trace(prefix_blocks, wrap_think=False),
            inserted_window=render_window(generated_blocks),
            next_original_steer=next_original_steer,
            next_original_exec_preview=next_original_exec_preview,
            mock=False,
        )
    else:
        judge = await maybe_call_bridge_judge(
            openrouter=None,
            prompts_dir=PROMPTS_DIR,
            trace_prefix=render_trace(prefix_blocks, wrap_think=False),
            inserted_window=render_window(generated_blocks),
            next_original_steer=next_original_steer,
            next_original_exec_preview=next_original_exec_preview,
            mock=True,
        )
    print(judge.model_dump())

## Compare all three steer variants for the same intervention

This is useful when you want to see whether one variant consistently sounds more natural than the others.

In [ ]:
for v in intervention_spec["variants"]:
    values = build_prompt_values(
        record=record,
        all_blocks=all_blocks,
        prefix_blocks=prefix_blocks,
        intervention_spec=intervention_spec,
        intervention_variant=v,
        pairs_to_generate_k=k_pairs,
        exec_token_limit=EXEC_TOKEN_LIMIT,
        style_window_pairs=STYLE_WINDOW_PAIRS,
        validation_feedback="",
    )
    rendered = render_prompt(prompt_path, values)
    print("=" * 100)
    print("VARIANT:", v)
    print(rendered[:1200], "..." if len(rendered) > 1200 else "")
    print()

## Batch-run all loaded records

This runs the current `records` slice in parallel. With the default configuration, that means 10 records.

In [ ]:
async def run_loaded_records_batch() -> list[dict[str, Any]]:
    """Augment the loaded notebook records concurrently and return per-record results."""
    batch_seed = 1234
    shared_client = None
    if OPENROUTER_API_KEY:
        shared_client = make_openrouter_client()

    async def process_record(
        record_index: int, record: Dict[str, Any]
    ) -> Dict[str, Any]:
        local_rng = random.Random(batch_seed + record_index)
        intervention_spec = choose_intervention(
            interventions_obj=interventions_obj,
            rng=local_rng,
            name=None,
        )
        try:
            augmented = await augment_record(
                record=record,
                intervention_spec=intervention_spec,
                prompts_dir=PROMPTS_DIR,
                rng=local_rng,
                token_counter=token_counter,
                exec_token_limit=EXEC_TOKEN_LIMIT,
                style_window_pairs=STYLE_WINDOW_PAIRS,
                openrouter=shared_client,
                mock_intervention=shared_client is None,
                wrap_think=False,
                vllm_base_url=None,
                vllm_model=None,
                max_attempts=3,
                run_bridge_judge=True,
            )
            augmentation_meta = augmented.get("augmentation", {})
            return {
                "record_index": record_index,
                "record_id": coalesce_record_task_id(record),
                "ok": True,
                "intervention_name": augmentation_meta.get("intervention_name", ""),
                "mode": augmentation_meta.get("editor_mode", ""),
                "suffix_decision": augmentation_meta.get("suffix_decision", ""),
                "pair_preview": augmentation_meta.get("pair_preview", {}),
                "generated_intervention_blocks": augmented.get(
                    "generated_intervention_blocks", []
                ),
                "augmented_prefix_trace": augmented["augmented_prefix_trace"],
            }
        except Exception as exc:
            return {
                "record_index": record_index,
                "record_id": coalesce_record_task_id(record),
                "ok": False,
                "error": str(exc),
            }

    try:
        batch_results = await asyncio.gather(
            *(
                process_record(record_index=index, record=record)
                for index, record in enumerate(records)
            )
        )
    finally:
        if shared_client is not None:
            await shared_client.aclose()

    return batch_results


batch_results = await run_loaded_records_batch()
print(f"Processed {len(batch_results)} records.")
print(f"Succeeded: {sum(1 for item in batch_results if item['ok'])}")
print(f"Failed: {sum(1 for item in batch_results if not item['ok'])}")
for item in batch_results:
    if item["ok"]:
        pair_preview = item.get("pair_preview", {})
        prefix_pairs = pair_preview.get("prefix_pairs", [])
        suffix_pair = pair_preview.get("suffix_pair_if_kept", [])

        print(
            item["record_index"],
            item["record_id"],
            item["intervention_name"],
            item["mode"],
            item["suffix_decision"],
        )
        if prefix_pairs:
            print("prefix tail (last 2 pairs):")
            print(render_window(prefix_pairs))
        print("generated intervention:")
        print(render_window(item["generated_intervention_blocks"]))
        if pair_preview.get("suffix_shown"):
            print("next suffix pair kept:")
            print(render_window(suffix_pair))
        print("-" * 80)
    else:
        print(item["record_index"], item["record_id"], item["error"])


## Notes

Prompt edits are easiest if you:
- change one thing at a time,
- keep the validation rules fixed,
- compare outputs on the same cut point,
- and inspect whether the inserted window preserves or intentionally redirects the local trajectory as intended.

Because the notebook and the batch script read the same prompt files, any improvement you make here can be used directly in the batch pipeline.